In [5]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['PINECONE_API_KEY']=os.getenv('PINECONE_API_KEY')

In [3]:
FILINGS = [
    {
        "company": "Apple",
        "ticker": "AAPL",
        "exchange": "NASDAQ",
        "filing_type": "10-Q",
        "period": "Q2-FY2026",
        "currency": "USD",
        "accounting_standard": "US GAAP",
        "source": "SEC EDGAR",
        "url": "https://www.sec.gov/Archives/edgar/data/320193/000032019326000013/aapl-20260328.htm",
        "local_path": "../data/raw/aapl_10q_q2_2026.pdf"
    },
    {
        "company": "Microsoft",
        "ticker": "MSFT",
        "exchange": "NASDAQ",
        "filing_type": "10-Q",
        "period": "Q1-FY2026",
        "currency": "USD",
        "accounting_standard": "US GAAP",
        "source": "SEC EDGAR",
        "url": "https://www.sec.gov/Archives/edgar/data/0000789019/000119312526027207/msft-20251231.htm",
        "local_path": "../data/raw/msft_10q_q1_2026.pdf"
    }
]

print(f"Filings registered: {len(FILINGS)}")
for f in FILINGS:
    print(f"  - {f['company']} ({f['ticker']}) | {f['filing_type']} | {f['period']} | {f['currency']}")

Filings registered: 2
  - Apple (AAPL) | 10-Q | Q2-FY2026 | USD
  - Microsoft (MSFT) | 10-Q | Q1-FY2026 | USD


In [4]:
# Re-convert PDFs
from docling.document_converter import DocumentConverter

converter = DocumentConverter()
all_results = {}

for filing in FILINGS:
    print(f"Converting {filing['company']}...")
    result = converter.convert(filing["local_path"])
    all_results[filing["ticker"]] = {
        "result": result,
        "metadata": {
            "company": filing["company"],
            "ticker": filing["ticker"],
            "period": filing["period"],
            "currency": filing["currency"]
        }
    }
    print(f"  ✓ {filing['ticker']} done")

print("\nAll converted. Now re-run extract_chunks cell.")

c:\Users\aksaj\Documents\ai-engineering\financial-reconciliation-agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Converting Apple...


[INFO] 2026-07-14 16:43:43,894 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-07-14 16:43:43,965 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-07-14 16:43:43,973 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.9.1/torch/PP-OCRv4/det/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-07-14 16:43:46,965 [RapidOCR] download_file.py:82: Download size: 13.83MB
[INFO] 2026-07-14 16:43:51,262 [RapidOCR] download_file.py:95: Successfully saved to: C:\Users\aksaj\Documents\ai-engineering\financial-reconciliation-agent\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-07-14 16:43:51,268 [RapidOCR] main.py:50: Using C:\Users\aksaj\Documents\ai-engineering\financial-reconciliation-agent\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-07-14 16:43:51,701 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-07-14 16:43:51,702 [RapidOCR] device_config.py:

  ✓ AAPL done
Converting Microsoft...
  ✓ MSFT done

All converted. Now re-run extract_chunks cell.


In [6]:
SECTION_PATTERNS = {
    "Income Statement": [
        "statements of operations",
        "statements of income",
        "income statements",
        "income statement"
    ],
    "Balance Sheet": [
        "balance sheet",
        "financial position",
        "balance sheets"
    ],
    "Cash Flow": [
        "cash flows",
        "cash flow statement",
        "cash flows statements"
    ],
    "Shareholders Equity": [
        "shareholders equity",
        "stockholders equity",
        "stockholders' equity statements",
        "equity statements"
    ],
    "MD&A": [
        "management's discussion",
        "results of operations",
        "summary results of operations"
    ],
    "Risk Factors": ["risk factors"],
    "Legal Proceedings": ["legal proceedings"],
    "Notes": [
        "notes to condensed",
        "notes to consolidated",
        "notes to financial statements"
    ],
    "Signature": ["pursuant to the requirements", "duly authorized"],
    "Exhibits": ["exhibit 31", "exhibit 32", "incorporated by reference"]
}

def detect_section(text):
    text_lower = text.lower()
    for section, patterns in SECTION_PATTERNS.items():
        for pattern in patterns:
            if pattern in text_lower:
                return section
    return "General"



In [7]:
from pydantic import BaseModel
from typing import Literal

class SectionClassification(BaseModel):
    section: Literal[
        "Income Statement", "Balance Sheet", "Cash Flow",
        "Shareholders Equity", "MD&A", "Risk Factors",
        "Legal Proceedings", "Notes", "Signature", "Exhibits", "General"
    ]
    confidence: Literal["high", "low"]

section_cache = {}

def detect_section_llm(header_text: str) -> SectionClassification:
    if header_text in section_cache:
        return section_cache[header_text]
    
    response = openai_client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": """You are a financial document section classifier for SEC 10-Q filings.
Classify section headers into the correct financial section.
Set confidence=high when header clearly maps to a financial section.
Set confidence=low when header is ambiguous, navigation text, or unclear.

Examples:
  "INCOME STATEMENTS" → Income Statement, high
  "BALANCE SHEETS" → Balance Sheet, high
  "MANAGEMENT'S DISCUSSION AND ANALYSIS" → MD&A, high
  "NOTES TO FINANCIAL STATEMENTS" → Notes, high
  "STOCKHOLDERS' EQUITY STATEMENTS" → Shareholders Equity, high
  "CASH FLOWS STATEMENTS" → Cash Flow, high
  "PART I Item 1" → General, low
  "Effective Tax Rate" → General, low
  "RISK FACTORS" → Risk Factors, high
  "LEGAL PROCEEDINGS" → Legal Proceedings, high"""
            },
            {"role": "user", "content": f"Classify this section header: '{header_text}'"}
        ],
        response_format=SectionClassification,
        temperature=0
    )
    
    result = response.choices[0].message.parsed
    section_cache[header_text] = result
    return result


def detect_section_smart(header_text: str) -> str:
    regex_result = detect_section(header_text)
    if regex_result != "General":
        return regex_result
    llm_result = detect_section_llm(header_text)
    if llm_result.confidence == "high":
        return llm_result.section
    return "General"


# Test on problem headers
test_headers = [
    "INCOME STATEMENTS",
    "BALANCE SHEETS",
    "STOCKHOLDERS' EQUITY STATEMENTS",
    "NOTES TO FINANCIAL STATEMENTS",
    "CASH FLOWS STATEMENTS",
    "PART I Item 1",
    "Effective Tax Rate",
    "MANAGEMENT'S DISCUSSION AND ANALYSIS OF FINANCIAL CONDITION"
]

print("Testing detect_section_smart:\n")
for header in test_headers:
    result = detect_section_smart(header)
    cache_hit = header in section_cache
    print(f"  '{header[:65]}'")
    print(f"  → {result} {'(LLM)' if cache_hit else '(regex)'}\n")

Testing detect_section_smart:

  'INCOME STATEMENTS'
  → Income Statement (regex)

  'BALANCE SHEETS'
  → Balance Sheet (regex)

  'STOCKHOLDERS' EQUITY STATEMENTS'
  → Shareholders Equity (regex)

  'NOTES TO FINANCIAL STATEMENTS'
  → Notes (regex)

  'CASH FLOWS STATEMENTS'
  → Cash Flow (regex)



NameError: name 'openai_client' is not defined

In [ ]:
from collections import Counter

all_parents = {}
all_children = {}

for filing in FILINGS:
    ticker = filing["ticker"]
    result = all_results[ticker]["result"]
    filing_metadata = {
        "company": filing["company"],
        "ticker": ticker,
        "period": filing["period"],
        "currency": filing["currency"]
    }
    
    parents, children = create_parent_child_chunks(
        ticker, result, filing_metadata
    )
    
    all_parents[ticker] = parents
    all_children[ticker] = children
    
    
    sections = Counter(c["metadata"]["section"] for c in children)
    print(f"\n{ticker}:")
    print(f"  Parents : {len(parents)}")
    print(f"  Children: {len(children)}")
    for section, count in sections.most_common():
        print(f"    {section}: {count}")


AAPL:
  Parents : 199
  Children: 447
    Notes: 124
    MD&A: 110
    Risk Factors: 48
    Balance Sheet: 47
    General: 39
    Income Statement: 37
    Cash Flow: 28
    Legal Proceedings: 14

MSFT:
  Parents : 468
  Children: 1011
    MD&A: 444
    Income Statement: 342
    Cash Flow: 93
    General: 38
    Balance Sheet: 34
    Notes: 34
    Shareholders Equity: 24
    Legal Proceedings: 1
    Risk Factors: 1


In [155]:
import json
import os

docstore = {}
for ticker, parents in all_parents.items():
    docstore.update(parents)

os.makedirs("../data/processed", exist_ok=True)
with open("../data/processed/docstore.json", "w") as f:
    json.dump(docstore, f)

print(f"Docstore saved: {len(docstore)} parents")

Docstore saved: 667 parents


In [156]:
all_child_texts = []
for ticker, children in all_children.items():
    for child in children:
        all_child_texts.append(child["content"])

bm25_encoder = BM25Encoder()
bm25_encoder.fit(all_child_texts)
bm25_encoder.dump("../data/processed/bm25_encoder.json")

print(f"BM25 fitted on {len(all_child_texts)} children")
print("BM25 saved")

  0%|          | 0/1458 [00:00<?, ?it/s]

BM25 fitted on 1458 children
BM25 saved


In [157]:
# Delete existing PC namespaces first
index.delete(delete_all=True, namespace="AAPL_PC")
index.delete(delete_all=True, namespace="MSFT_PC")
print("Old namespaces cleared")

# Re-upsert
for ticker, children in all_children.items():
    upsert_children_hybrid(ticker, children)

print("\nFinal index stats:")
print(index.describe_index_stats())

Old namespaces cleared

Upserting AAPL children to namespace: AAPL_PC
Total children: 447


100%|██████████| 5/5 [00:20<00:00,  4.09s/it]


  ✓ AAPL children upserted | skipped: 2

Upserting MSFT children to namespace: MSFT_PC
Total children: 1011


100%|██████████| 11/11 [00:34<00:00,  3.10s/it]


  ✓ MSFT children upserted | skipped: 1

Final index stats:
{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'dotproduct',
 'namespaces': {'AAPL': {'vector_count': 265},
                'AAPL_PC': {'vector_count': 445},
                'MSFT': {'vector_count': 662},
                'MSFT_PC': {'vector_count': 1010}},
 'total_vector_count': 2382,
 'vector_type': 'dense'}


In [150]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from docling.datamodel.base_models import DocItemLabel

CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

def get_page_no(element):
    try:
        return element.prov[0].page_no
    except:
        return None

def extract_chunks(ticker, result, filing_metadata):
    chunks = []
    current_section = "General"
    chunk_index = 0
    doc = result.document

    for element, level in doc.iterate_items():
        # Get page number
        page_no = get_page_no(element)

        # Skip page headers and footers
        if hasattr(element, 'label') and element.label in [
            DocItemLabel.PAGE_HEADER, 
            DocItemLabel.PAGE_FOOTER
        ]:
            continue

        # Section detection from headers
        if hasattr(element, 'label') and element.label in [
            DocItemLabel.SECTION_HEADER, DocItemLabel.TITLE
        ]:
            if hasattr(element, 'text'):
                detected = detect_section(element.text)
                if detected != "General":
                    current_section = detected
            continue

        # Tables — atomic chunk
        if hasattr(element, 'data'):
            table_text = element.export_to_markdown(doc=doc)
            if table_text.strip():
                chunks.append({
                    "content": table_text,
                    "metadata": {
                        **filing_metadata,
                        "chunk_index": chunk_index,
                        "chunk_type": "table",
                        "section": current_section,
                        "page": page_no
                    }
                })
                chunk_index += 1
            continue

        # Prose — split
        if hasattr(element, 'text') and element.text.strip():
            
            splits = splitter.split_text(element.text.strip())
            for split in splits:
                chunks.append({
                    "content": split,
                    "metadata": {
                        **filing_metadata,
                        "chunk_index": chunk_index,
                        "chunk_type": "prose",
                        "section": current_section,
                        "page": page_no
                    }
                })
                chunk_index += 1

    return chunks


all_chunks = {}
for filing in FILINGS:
    ticker = filing["ticker"]
    result = all_results[ticker]["result"]
    filing_metadata = all_results[ticker]["metadata"]
    chunks = extract_chunks(ticker, result, filing_metadata)
    all_chunks[ticker] = chunks
    print(f"{ticker}: {len(chunks)} chunks — prose + tables separated")

AAPL: 267 chunks — prose + tables separated
MSFT: 663 chunks — prose + tables separated


In [82]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

INDEX_NAME = "financial-reconciliation"
EMBEDDING_DIM = 1536  # text-embedding-3-small

# Create index if it doesn't exist
if pc.has_index(INDEX_NAME):  
    pc.delete_index(name=INDEX_NAME) 

existing_indexes = [idx.name for idx in pc.list_indexes()]

if INDEX_NAME not in existing_indexes:
    print(f"Creating index: {INDEX_NAME}")
    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBEDDING_DIM,
        metric="dotproduct",  # required for hybrid search
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print("Index created")
else:
    print(f"Index already exists: {INDEX_NAME}")

index = pc.Index(INDEX_NAME)
print(f"\nIndex stats: {index.describe_index_stats()}")

Creating index: financial-reconciliation
Index created

Index stats: {'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'dotproduct',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}


In [143]:
from pinecone_text.sparse import BM25Encoder

# Collect all texts from all chunks
all_texts = []
for ticker, chunks in all_chunks.items():
    for chunk in chunks:
        all_texts.append(chunk["content"])

print(f"Total texts to fit BM25: {len(all_texts)}")

# Fit BM25 on our corpus
bm25_encoder = BM25Encoder()
bm25_encoder.fit(all_texts)

print("BM25Encoder fitted successfully")

Total texts to fit BM25: 930


  0%|          | 0/930 [00:00<?, ?it/s]

BM25Encoder fitted successfully


In [84]:
# Embed chunks and upsert to Pinecone

from openai import OpenAI
import hashlib
from tqdm import tqdm

openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
EMBEDDING_MODEL = "text-embedding-3-small"

# We send 100 chunks at a time to OpenAI, not all at once
# Two reasons: 
# Rate limits - OpenAI has a rate limit on embedding API calls.
#               Sending all  chunks at once might hit the limit. Batches of 100 are safe.
# Memory -      Sending all embeddings in one API call means holding all  × * 1536 numbers in memory simultaneously.
#                Batches keep memory usage low.
BATCH_SIZE = 100





def get_embeddings(texts):
    # Input:  ["Total net sales $111,184", "iPhone net sales increased..."]
    # Output: [[0.023, -0.14, 0.89, ...1536 numbers], [0.11, 0.03, -0.44, ...1536 numbers]]
    response = openai_client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=texts
    )
    return [e.embedding for e in response.data]

def make_chunk_id(ticker, chunk_index):
    # Creates a unique ID for every chunk
    # Pinecone IDs must be consistent, short, and collision-free
    # MD5 gives us a fixed 16 character string regardless of input length
    # if we ever re-run ingestion, the same chunk gets the same ID : Avoids duplicates
    raw = f"{ticker}_{chunk_index}"
    return hashlib.md5(raw.encode()).hexdigest()[:16]

def upsert_chunks_hybrid(ticker, chunks):
    print(f"\nUpserting {ticker}: {len(chunks)} chunks (hybrid)")

    skipped = 0
    for i in tqdm(range(0, len(chunks), BATCH_SIZE)):
        batch = chunks[i:i + BATCH_SIZE]
        texts = [c["content"] for c in batch]
        
        # Dense vectors
        dense_embeddings = get_embeddings(texts)
        
        # Sparse vectors (BM25)
        sparse_embeddings = bm25_encoder.encode_documents(texts)
        
        vectors = []
        for chunk, dense, sparse in zip(batch, dense_embeddings, sparse_embeddings):
            # Skip chunks with empty sparse vectors
            if not sparse["values"]:
                skipped += 1
                continue

            chunk_id = make_chunk_id(ticker, chunk["metadata"]["chunk_index"])
            vectors.append({
                "id": chunk_id,
                "values": dense,  # ← dense
                "sparse_values": sparse, # ← BM25 sparse
                "metadata": {
                    **chunk["metadata"],
                    "content": chunk["content"][:1000]
                }
            })
        
        index.upsert(vectors=vectors, namespace=ticker)
    
    print(f"  ✓ {ticker} upserted with hybrid vectors")


# Re-upsert all
for ticker, chunks in all_chunks.items():
    upsert_chunks_hybrid(ticker, chunks)

print("\nFinal index stats:")
print(index.describe_index_stats())


Upserting AAPL: 267 chunks (hybrid)


100%|██████████| 3/3 [00:13<00:00,  4.53s/it]


  ✓ AAPL upserted with hybrid vectors

Upserting MSFT: 663 chunks (hybrid)


100%|██████████| 7/7 [00:22<00:00,  3.28s/it]


  ✓ MSFT upserted with hybrid vectors

Final index stats:
{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'dotproduct',
 'namespaces': {'AAPL': {'vector_count': 265}, 'MSFT': {'vector_count': 662}},
 'total_vector_count': 927,
 'vector_type': 'dense'}


In [87]:
NOISE_SECTIONS = ["General", "Signature", "Exhibits"]

def scale_dense(vector, alpha):
    return [v * alpha for v in vector]

def scale_sparse(vector, alpha):
    return {
        "indices": vector["indices"],
        "values": [v * (1 - alpha) for v in vector["values"]]
    }
    

def hybrid_retrieve_financial(query, ticker, section=None, chunk_type=None, top_k=5, alpha=0.5):
    dense_vector = get_embeddings([query])[0]
    sparse_vector = bm25_encoder.encode_queries(query)
    
    # Build filter
    filter_dict = {"section": {"$nin": NOISE_SECTIONS}}
    
    if section:
        filter_dict["section"] = {"$eq": section}
    else:
        filter_dict["section"] = {"$nin": NOISE_SECTIONS}
    
    results = index.query(
        vector=scale_dense(dense_vector, alpha),
        sparse_vector=scale_sparse(sparse_vector, alpha),
        top_k=top_k,
        namespace=ticker,
        include_metadata=True,
        filter=filter_dict
    )
    
    return results.matches


# Test — filter to Income Statement tables only
print("TEST 1 — Filtered to Income Statement tables:")
results = hybrid_retrieve_financial(
    "What was Apple total net sales in Q2 2026?",
    "AAPL",
    section="Income Statement",
    chunk_type="table",
    top_k=3
)

for i, match in enumerate(results, 1):
    print(f"Result {i}: section={match.metadata.get('section')} | type={match.metadata.get('chunk_type')} | page={match.metadata.get('page')} | score={match.score:.4f}")
    print(f"  Content: {match.metadata.get('content')[:300]}")
    print()

TEST 1 — Filtered to Income Statement tables:
Result 1: section=Income Statement | type=table | page=4.0 | score=0.3599
  Content: |                                              | Three Months Ended   | Three Months Ended   | Six Months Ended   | Six Months Ended   |
|----------------------------------------------|----------------------|----------------------|--------------------|--------------------|
|                         

Result 2: section=Income Statement | type=table | page=5.0 | score=0.2854
  Content: |                                                                              | Three Months Ended   | Three Months Ended   | Six Months Ended   | Six Months Ended   |
|------------------------------------------------------------------------------|----------------------|----------------------|-----

Result 3: section=Income Statement | type=prose | page=5.0 | score=0.1827
  Content: (In millions)



In [121]:
GOLDEN_QUERIES = [
  {
    "id": "Q001",
    "ticker": "AAPL",
    "query": "What was Apple's total net sales for the three months ended March 28, 2026, and how does it compare to the same quarter last year?",
    "category": "single",
    "expected_section": "Income Statement",
    "expected_chunk_type": "table",
    "answer_hint": "$111,184M in Q2 2026 vs $95,359M in Q2 2025, a 17% increase."
  },
  {
    "id": "Q002",
    "ticker": "AAPL",
    "query": "What was diluted earnings per share for Q2 2026 versus Q2 2025?",
    "category": "single",
    "expected_section": "Income Statement",
    "expected_chunk_type": "table",
    "answer_hint": "$2.01 in Q2 2026 vs $1.65 in Q2 2025."
  },
  {
    "id": "Q003",
    "ticker": "AAPL",
    "query": "What was Apple's gross margin in dollars for the three months ended March 28, 2026, compared to the same period in 2025?",
    "category": "single",
    "expected_section": "Income Statement",
    "expected_chunk_type": "table",
    "answer_hint": "$54,781M in Q2 2026 vs $44,867M in Q2 2025 (both figures appear on the same income statement line)."
  },
  {
    "id": "Q004",
    "ticker": "AAPL",
    "query": "What was net income for the six-month period ended March 28, 2026?",
    "category": "single",
    "expected_section": "Income Statement",
    "expected_chunk_type": "table",
    "answer_hint": "$71,675M vs $61,110M in the prior-year six-month period."
  },
  {
    "id": "Q005",
    "ticker": "AAPL",
    "query": "What was Apple's effective tax rate for Q2 2026, and what reasons does MD&A give for it being below the statutory federal rate?",
    "category": "single",
    "expected_section": "MD&A",
    "expected_chunk_type": "table + prose",
    "answer_hint": "17.5% effective rate vs 21% statutory. MD&A's Provision for Income Taxes subsection attributes this to a lower effective tax rate on foreign earnings, the federal R&D credit, and a change in valuation allowance, partly offset by state income taxes \u2014 both the number and explanation sit in this one MD&A subsection."
  },
  {
    "id": "Q006",
    "ticker": "AAPL",
    "query": "What were iPhone net sales in Q2 2026, and what was the year-over-year percentage change according to MD&A?",
    "category": "single",
    "expected_section": "MD&A",
    "expected_chunk_type": "table",
    "answer_hint": "$56,994M, up 22% from $46,841M \u2014 both the dollar figure and the % change appear together in MD&A's Products and Services Performance table."
  },
  {
    "id": "Q007",
    "ticker": "AAPL",
    "query": "How much did Services net sales grow over the first six months of fiscal 2026?",
    "category": "single",
    "expected_section": "MD&A",
    "expected_chunk_type": "table",
    "answer_hint": "$60,989M vs $52,985M, a 15% increase."
  },
  {
    "id": "Q008",
    "ticker": "AAPL",
    "query": "What percentage of total net sales did R&D expense represent in Q2 2026?",
    "category": "single",
    "expected_section": "MD&A",
    "expected_chunk_type": "table",
    "answer_hint": "10% of total net sales, up from 9% in Q2 2025."
  },
  {
    "id": "Q009",
    "ticker": "AAPL",
    "query": "What was the weighted-average diluted share count used to compute EPS for the six months ended March 28, 2026?",
    "category": "single",
    "expected_section": "Notes",
    "expected_chunk_type": "table",
    "answer_hint": "14,768,115 thousand diluted weighted-average shares, per the EPS note's computation table."
  },
  {
    "id": "Q010",
    "ticker": "AAPL",
    "query": "How much did total operating expenses increase in dollar terms and percentage terms during Q2 2026 versus Q2 2025?",
    "category": "single",
    "expected_section": "MD&A",
    "expected_chunk_type": "table",
    "answer_hint": "$18,896M vs $15,278M, a 24% increase."
  },
  {
    "id": "Q011",
    "ticker": "AAPL",
    "query": "What was Apple's total assets figure as of March 28, 2026, compared to fiscal year-end September 27, 2025?",
    "category": "single",
    "expected_section": "Balance Sheet",
    "expected_chunk_type": "table",
    "answer_hint": "$371,082M at March 28, 2026 vs $359,241M at September 27, 2025."
  },
  {
    "id": "Q012",
    "ticker": "AAPL",
    "query": "What was Apple's cash and cash equivalents balance at the end of Q2 2026?",
    "category": "single",
    "expected_section": "Balance Sheet",
    "expected_chunk_type": "table",
    "answer_hint": "$45,572M, up from $35,934M at fiscal year-end 2025."
  },
  {
    "id": "Q013",
    "ticker": "AAPL",
    "query": "How much commercial paper did Apple have outstanding at the end of Q2 2026?",
    "category": "single",
    "expected_section": "Balance Sheet",
    "expected_chunk_type": "table",
    "answer_hint": "$1,997M, down from $7,979M at fiscal year-end 2025."
  },
  {
    "id": "Q014",
    "ticker": "AAPL",
    "query": "What was total shareholders' equity as of March 28, 2026, and how does that compare to fiscal year-end 2025?",
    "category": "single",
    "expected_section": "Balance Sheet",
    "expected_chunk_type": "table",
    "answer_hint": "$106,491M vs $73,733M \u2014 a large increase driven largely by net income and share-based comp exceeding buybacks and dividends."
  },
  {
    "id": "Q015",
    "ticker": "AAPL",
    "query": "According to the statement of shareholders' equity, what drove the change in retained earnings during the six months ended March 28, 2026?",
    "category": "single",
    "expected_section": "Shareholders' Equity",
    "expected_chunk_type": "table",
    "answer_hint": "Retained earnings moved from $(14,264)M to $12,359M, driven by $71,675M net income, partly offset by $(7,735)M dividends, $(1,024)M net share settlement withholding, and $(36,293)M of share repurchases \u2014 all shown in this one statement."
  },
  {
    "id": "Q016",
    "ticker": "AAPL",
    "query": "What is the current portion of term debt on Apple's balance sheet as of March 28, 2026?",
    "category": "single",
    "expected_section": "Balance Sheet",
    "expected_chunk_type": "table",
    "answer_hint": "$8,310M, down from $12,350M at fiscal year-end 2025."
  },
  {
    "id": "Q017",
    "ticker": "AAPL",
    "query": "How did Apple's inventories change from fiscal year-end 2025 to Q2 2026?",
    "category": "single",
    "expected_section": "Balance Sheet",
    "expected_chunk_type": "table",
    "answer_hint": "Increased from $5,718M to $6,747M, a rise of about $1,029M."
  },
  {
    "id": "Q018",
    "ticker": "AAPL",
    "query": "What is the aggregate carrying amount and fair value of Apple's outstanding term debt (Notes) as of March 28, 2026?",
    "category": "single",
    "expected_section": "Notes",
    "expected_chunk_type": "table",
    "answer_hint": "Carrying amount of $82.7 billion and fair value of $70.8 billion, both disclosed in the Term Debt note."
  },
  {
    "id": "Q019",
    "ticker": "AAPL",
    "query": "How much cash was generated by operating activities during the six months ended March 28, 2026?",
    "category": "single",
    "expected_section": "Cash Flow",
    "expected_chunk_type": "table",
    "answer_hint": "$82,627M, up from $53,887M in the prior-year six-month period."
  },
  {
    "id": "Q020",
    "ticker": "AAPL",
    "query": "How much cash did Apple use for share repurchases during the six months ended March 28, 2026, per the cash flow statement?",
    "category": "single",
    "expected_section": "Cash Flow",
    "expected_chunk_type": "table",
    "answer_hint": "$36,989M, per the financing activities section of the cash flow statement."
  },
  {
    "id": "Q021",
    "ticker": "AAPL",
    "query": "What was Apple's capital expenditure (payments for property, plant and equipment) for the six months ended March 28, 2026, versus the prior year?",
    "category": "single",
    "expected_section": "Cash Flow",
    "expected_chunk_type": "table",
    "answer_hint": "$4,344M in H1 2026 vs $6,011M in H1 2025 \u2014 a year-over-year decrease."
  },
  {
    "id": "Q022",
    "ticker": "AAPL",
    "query": "Did investing activities use or generate cash during the first six months of fiscal 2026, and how does that compare to the prior-year period? (Edge case: sign flips between periods.)",
    "category": "single",
    "expected_section": "Cash Flow",
    "expected_chunk_type": "table",
    "answer_hint": "Investing activities used $11,054M of cash in H1 2026, versus generating $12,709M in H1 2025 \u2014 a swing from a net source to a net use of cash."
  },
  {
    "id": "Q023",
    "ticker": "AAPL",
    "query": "How much did Apple pay in dividends and dividend equivalents during the six months ended March 28, 2026?",
    "category": "single",
    "expected_section": "Cash Flow",
    "expected_chunk_type": "table",
    "answer_hint": "$7,743M, up slightly from $7,614M in the prior-year period."
  },
  {
    "id": "Q024",
    "ticker": "AAPL",
    "query": "What was cash paid for income taxes, net, for the six months ended March 28, 2026, compared to the prior-year period?",
    "category": "single",
    "expected_section": "Cash Flow",
    "expected_chunk_type": "table",
    "answer_hint": "$20,397M in H1 2026 vs $31,683M in H1 2025, per the supplemental cash flow disclosure line."
  },
  {
    "id": "Q025",
    "ticker": "AAPL",
    "query": "What was the ending cash, cash equivalents, and restricted cash balance shown on Apple's cash flow statement for the six months ended March 28, 2026?",
    "category": "single",
    "expected_section": "Cash Flow",
    "expected_chunk_type": "table",
    "answer_hint": "$45,572M, per the ending balance line of the cash flow statement."
  },
  {
    "id": "Q026",
    "ticker": "AAPL",
    "query": "Why did iPhone net sales increase during Q2 2026 compared to Q2 2025?",
    "category": "single",
    "expected_section": "MD&A",
    "expected_chunk_type": "prose",
    "answer_hint": "MD&A attributes the increase to higher net sales of Pro models."
  },
  {
    "id": "Q027",
    "ticker": "AAPL",
    "query": "Why did Services gross margin percentage increase during the first six months of fiscal 2026?",
    "category": "single",
    "expected_section": "MD&A",
    "expected_chunk_type": "prose",
    "answer_hint": "Primarily due to a different mix of services and strength in foreign currencies relative to the U.S. dollar, partially offset by higher costs."
  },
  {
    "id": "Q028",
    "ticker": "AAPL",
    "query": "Why was Apple's effective tax rate for the six months ended March 28, 2026 lower than the statutory federal rate, and why was it higher than the same period a year earlier?",
    "category": "single",
    "expected_section": "MD&A",
    "expected_chunk_type": "prose",
    "answer_hint": "Lower than statutory due to lower tax on foreign earnings, the federal R&D credit, and share-based compensation tax benefits (offset by state taxes). Higher year-over-year mainly due to changes in unrecognized tax benefits and other prior-year specific items."
  },
  {
    "id": "Q029",
    "ticker": "AAPL",
    "query": "Apple announced a new MacBook Neo during Q2 2026 \u2014 did MD&A attribute any of the reported change in Mac segment net sales to this or other new products? (Cross-section: product-introduction list vs. segment performance narrative.)",
    "category": "single",
    "expected_section": "MD&A",
    "expected_chunk_type": "prose",
    "answer_hint": "MD&A's Mac discussion attributes the Q2 2026 increase generally to higher net sales of laptops, without singling out MacBook Neo by name; the six-month period was described as relatively flat year over year despite the product introduction."
  },
  {
    "id": "Q030",
    "ticker": "AAPL",
    "query": "Why did Greater China net sales increase during Q2 2026?",
    "category": "single",
    "expected_section": "MD&A",
    "expected_chunk_type": "prose",
    "answer_hint": "Due to higher net sales of iPhone, with a favorable year-over-year impact from the strength of the renminbi relative to the U.S. dollar."
  },
  {
    "id": "Q031",
    "ticker": "AAPL",
    "query": "What risks does Apple disclose related to its use of artificial intelligence in products and services?",
    "category": "single",
    "expected_section": "Risk Factors",
    "expected_chunk_type": "prose",
    "answer_hint": "Risks cited include competition and strategy, cost recoupment, product liability, IP infringement, data privacy, cybersecurity, sanctions/export controls, exposure to harmful or inaccurate content, bias and discrimination, and online safety/protection of minors."
  },
  {
    "id": "Q032",
    "ticker": "AAPL",
    "query": "What cybersecurity risks does Apple highlight regarding confidential information and AI-enabled attacks?",
    "category": "single",
    "expected_section": "Risk Factors",
    "expected_chunk_type": "prose",
    "answer_hint": "Apple discloses risk of unauthorized access to confidential/personal data, reliance on suppliers also exposed to attacks, and notes AI technologies are being used to produce targeted phishing, automate vulnerability discovery, and generate deepfakes that can bypass authentication."
  },
  {
    "id": "Q033",
    "ticker": "AAPL",
    "query": "What risk does Apple disclose regarding design and manufacturing defects in its products?",
    "category": "single",
    "expected_section": "Risk Factors",
    "expected_chunk_type": "prose",
    "answer_hint": "Defects in hardware, software, or components (including third-party components) can create safety, environmental, or liability risks, and can lead to recalls, warranty costs, litigation, regulatory fines, and reputational harm."
  },
  {
    "id": "Q034",
    "ticker": "AAPL",
    "query": "What is the status of the EU Digital Markets Act investigations against Apple, and what fine has been imposed so far?",
    "category": "single",
    "expected_section": "Legal Proceedings",
    "expected_chunk_type": "prose",
    "answer_hint": "The European Commission fined Apple \u20ac500 million in the Article 5(4) Investigation (which Apple is appealing) and issued preliminary findings in a separate ongoing Article 6(4) Investigation that could result in fines up to 10% of Apple's annual worldwide net sales."
  },
  {
    "id": "Q035",
    "ticker": "AAPL",
    "query": "What is the current status of the Epic Games litigation and the App Store injunction?",
    "category": "single",
    "expected_section": "Legal Proceedings",
    "expected_chunk_type": "prose",
    "answer_hint": "The Ninth Circuit upheld the 2025 Injunction in part, modifying it to allow Apple to require size/placement parity for external purchase links and to charge a commission on link-out purchases, and remanded the matter to the district court."
  }
]

print(f"Golden queries: {len(GOLDEN_QUERIES)}")
print("\nBreakdown:")
from collections import Counter
sections = Counter(q["expected_section"] for q in GOLDEN_QUERIES)
types = Counter(q["expected_chunk_type"] for q in GOLDEN_QUERIES)
tickers = Counter(q["ticker"] for q in GOLDEN_QUERIES)

for section, count in sections.items():
    print(f"  {section}: {count}")
print()
for t, count in types.items():
    print(f"  {t}: {count}")
print()
for ticker, count in tickers.items():
    print(f"  {ticker}: {count}")

Golden queries: 35

Breakdown:
  Income Statement: 4
  MD&A: 10
  Notes: 2
  Balance Sheet: 6
  Shareholders' Equity: 1
  Cash Flow: 7
  Risk Factors: 3
  Legal Proceedings: 2

  table: 24
  table + prose: 1
  prose: 10

  AAPL: 35


In [100]:
def evaluate_retrieval(golden_queries, top_k=5, alpha=0.5):
    
    results = []
    
    for gq in golden_queries:
        # Retrieve
        matches = hybrid_retrieve_financial(
            query=gq["query"],
            ticker=gq["ticker"],
            top_k=top_k,
            alpha=alpha
        )
        
        # Check each result
        hit = False
        reciprocal_rank = 0
        relevant_count = 0
        
        for rank, match in enumerate(matches, 1):
            section_match = match.metadata.get("section") == gq["expected_section"]
            type_match = match.metadata.get("chunk_type") == gq["expected_chunk_type"]
            
            is_relevant = section_match and type_match
            
            if is_relevant:
                relevant_count += 1
                if not hit:
                    hit = True
                    reciprocal_rank = 1 / rank
        
        precision_at_k = relevant_count / top_k
        
        results.append({
            "id": gq["id"],
            "query": gq["query"],
            "ticker": gq["ticker"],
            "expected_section": gq["expected_section"],
            "expected_type": gq["expected_chunk_type"],
            "hit": hit,
            "reciprocal_rank": reciprocal_rank,
            "precision_at_k": precision_at_k
        })
    
    # Aggregate metrics
    hit_rate = sum(r["hit"] for r in results) / len(results)
    mrr = sum(r["reciprocal_rank"] for r in results) / len(results)
    avg_precision = sum(r["precision_at_k"] for r in results) / len(results)
    
    return results, {
        "hit_rate": round(hit_rate, 4),
        "mrr": round(mrr, 4),
        "avg_precision_at_k": round(avg_precision, 4),
        "top_k": top_k,
        "total_queries": len(results)
    }


# Run baseline evaluation
print("Running baseline evaluation...")
print("Strategy: Hybrid BM25 + Dense, no Query Router, no section filter\n")

results, metrics = evaluate_retrieval(GOLDEN_QUERIES, top_k=5, alpha=0.5)

print("="*50)
print("BASELINE METRICS")
print("="*50)
print(f"Hit Rate@{metrics['top_k']}     : {metrics['hit_rate']}")
print(f"MRR              : {metrics['mrr']}")
print(f"Avg Precision@{metrics['top_k']} : {metrics['avg_precision_at_k']}")
print(f"Total queries    : {metrics['total_queries']}")

print("\nPer query breakdown:")
print("-"*50)
for r in results:
    status = "✓" if r["hit"] else "✗"
    print(f"{status} {r['id']} | {r['ticker']} | {r['expected_section']} | RR={r['reciprocal_rank']:.2f} | P@k={r['precision_at_k']:.2f}")

Running baseline evaluation...
Strategy: Hybrid BM25 + Dense, no Query Router, no section filter

BASELINE METRICS
Hit Rate@5     : 0.4286
MRR              : 0.3533
Avg Precision@5 : 0.2171
Total queries    : 35

Per query breakdown:
--------------------------------------------------
✗ Q001 | AAPL | Income Statement | RR=0.00 | P@k=0.00
✓ Q002 | AAPL | Income Statement | RR=0.33 | P@k=0.20
✗ Q003 | AAPL | Income Statement / MD&A | RR=0.00 | P@k=0.00
✗ Q004 | AAPL | Income Statement | RR=0.00 | P@k=0.00
✗ Q005 | AAPL | MD&A / Income Statement Notes | RR=0.00 | P@k=0.00
✗ Q006 | AAPL | Notes (Revenue) / MD&A | RR=0.00 | P@k=0.00
✗ Q007 | AAPL | MD&A | RR=0.00 | P@k=0.00
✓ Q008 | AAPL | MD&A | RR=0.50 | P@k=0.20
✗ Q009 | AAPL | Cover Page / Balance Sheet / Notes (EPS) | RR=0.00 | P@k=0.00
✗ Q010 | AAPL | MD&A | RR=0.00 | P@k=0.00
✗ Q011 | AAPL | Balance Sheet | RR=0.00 | P@k=0.00
✓ Q012 | AAPL | Balance Sheet | RR=0.20 | P@k=0.20
✗ Q013 | AAPL | Balance Sheet / Notes (Debt) | RR=0.00 | P@

In [132]:
prompt = """You are a financial document query router for SEC 10-Q filings.

Given a user question, return a JSON object with exactly two fields:
1. "chunk_type": either "table" or "prose"
2. "section": one of these exact values:
   - "Income Statement" (revenue, net sales, gross margin, operating income, net income, EPS, cost of sales, R&D expense raw dollar amount, selling general administrative)
   - "Balance Sheet" (total assets, cash, liabilities, equity, receivables, inventory, inventories)
   - "Cash Flow" (operating activities, investing activities, financing activities, cash generated, dividends paid, capital expenditure)
   - "MD&A" (why metrics changed, segment performance, growth reasons, management discussion, percentage change, ratios of expenses to sales)
   - "Risk Factors" (risks, uncertainties, challenges)
   - "Legal Proceedings" (lawsuits, investigations, legal matters)
   - "Notes" (share repurchase, RSUs, debt details, accounting policies, commitments, EPS computation, weighted average shares)
   - "Shareholders Equity" (retained earnings change, AOCI, equity statement details)
   - "Any" (if question spans multiple sections)

PRIORITY RULES — apply these first:
- Question contains "inventor" (inventory/inventories) → ALWAYS Balance Sheet, table
- Question asks about dividends PAID as cash outflow → ALWAYS Cash Flow, table
- Question asks for R&D as percentage or ratio → ALWAYS MD&A, prose
- Question asks for EPS computation or weighted average shares → ALWAYS Notes, table

GENERAL RULES:
- Specific NUMBER or FIGURE → chunk_type = "table"
- EXPLANATION, REASON, WHY, HOW it changed → chunk_type = "prose"
- Even if question includes "compare" or "how does it compare" — classify by PRIMARY subject
  Example: "What was net sales and how does it compare?" → primary = net sales → Income Statement, table
- Geographic segment results → MD&A
- Effective tax rate explanation → MD&A, prose

Return ONLY valid JSON. No explanation. No markdown.
Example: {"chunk_type": "table", "section": "Income Statement"}"""

In [133]:
import json

def route_query(query):
    system_prompt = prompt

    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query}
        ],
        temperature=0
    )
    
    try:
        return json.loads(response.choices[0].message.content)
    except:
        # Fallback if JSON parsing fails
        return {"chunk_type": "prose", "section": "Any"}


# Test on a few queries
test_queries = [
    "What was Apple's effective tax rate for Q2 2026, and what reasons does MD&A give?",
    "What percentage of total net sales did R&D expense represent in Q2 2026?",
    "What was total shareholders equity as of March 28, 2026?",
    "How did Apple inventories change from fiscal year-end 2025 to Q2 2026?",
    "How much did Apple pay in dividends during the six months ended March 28, 2026?",
    "What was Apple total net sales and how does it compare to same quarter last year?"
]

print("Router test after fixes:\n")
for q in test_queries:
    route = route_query(q)
    print(f"Q: {q[:70]}...")
    print(f"   → {route}")
    print()

Router test after fixes:

Q: What was Apple's effective tax rate for Q2 2026, and what reasons does...
   → {'chunk_type': 'prose', 'section': 'MD&A'}

Q: What percentage of total net sales did R&D expense represent in Q2 202...
   → {'chunk_type': 'prose', 'section': 'MD&A'}

Q: What was total shareholders equity as of March 28, 2026?...
   → {'chunk_type': 'table', 'section': 'Shareholders Equity'}

Q: How did Apple inventories change from fiscal year-end 2025 to Q2 2026?...
   → {'chunk_type': 'prose', 'section': 'MD&A'}

Q: How much did Apple pay in dividends during the six months ended March ...
   → {'chunk_type': 'table', 'section': 'Cash Flow'}

Q: What was Apple total net sales and how does it compare to same quarter...
   → {'chunk_type': 'table', 'section': 'Income Statement'}



In [92]:
def retrieve_with_router(query, ticker, top_k=5, alpha=0.5):
    # Step 1 — Route the query
    route = route_query(query)
    section = route["section"]
    chunk_type = route["chunk_type"]
    
    # Step 2 — Retrieve with auto filters
    # If section is "Any" — no section filter
    if section == "Any":
        matches = hybrid_retrieve_financial(
            query=query,
            ticker=ticker,
            chunk_type=chunk_type,
            top_k=top_k,
            alpha=alpha
        )
    else:
        matches = hybrid_retrieve_financial(
            query=query,
            ticker=ticker,
            section=section,
            chunk_type=chunk_type,
            top_k=top_k,
            alpha=alpha
        )
    
    return matches, route


# Test on our problem query
query = "What was Apple total net sales in Q2 2026?"
matches, route = retrieve_with_router(query, "AAPL", top_k=3)

print(f"Query: {query}")
print(f"Route: {route}\n")
for i, match in enumerate(matches, 1):
    print(f"Result {i}: section={match.metadata.get('section')} | type={match.metadata.get('chunk_type')} | page={match.metadata.get('page')} | score={match.score:.4f}")
    print(f"  Content: {match.metadata.get('content')[:200]}")
    print()

Query: What was Apple total net sales in Q2 2026?
Route: {'chunk_type': 'table', 'section': 'Income Statement'}

Result 1: section=Income Statement | type=table | page=4.0 | score=0.3599
  Content: |                                              | Three Months Ended   | Three Months Ended   | Six Months Ended   | Six Months Ended   |
|----------------------------------------------|---------------

Result 2: section=Income Statement | type=table | page=5.0 | score=0.2854
  Content: |                                                                              | Three Months Ended   | Three Months Ended   | Six Months Ended   | Six Months Ended   |
|------------------------------

Result 3: section=Income Statement | type=prose | page=5.0 | score=0.1827
  Content: (In millions)



In [104]:
def evaluate_retrieval_with_router(golden_queries, top_k=5, alpha=0.5):
    
    results = []
    
    for gq in golden_queries:
        # Retrieve with router
        matches, route = retrieve_with_router(
            query=gq["query"],
            ticker=gq["ticker"],
            top_k=top_k,
            alpha=alpha
        )
        
        hit = False
        reciprocal_rank = 0
        relevant_count = 0
        
        for rank, match in enumerate(matches, 1):
            section_match = match.metadata.get("section") == gq["expected_section"]
            type_match = match.metadata.get("chunk_type") == gq["expected_chunk_type"]
            
            is_relevant = section_match and type_match
            
            if is_relevant:
                relevant_count += 1
                if not hit:
                    hit = True
                    reciprocal_rank = 1 / rank
        
        precision_at_k = relevant_count / top_k
        
        results.append({
            "id": gq["id"],
            "query": gq["query"],
            "ticker": gq["ticker"],
            "expected_section": gq["expected_section"],
            "expected_type": gq["expected_chunk_type"],
            "routed_section": route["section"],
            "routed_type": route["chunk_type"],
            "hit": hit,
            "reciprocal_rank": reciprocal_rank,
            "precision_at_k": precision_at_k
        })
    
    hit_rate = sum(r["hit"] for r in results) / len(results)
    mrr = sum(r["reciprocal_rank"] for r in results) / len(results)
    avg_precision = sum(r["precision_at_k"] for r in results) / len(results)
    
    return results, {
        "hit_rate": round(hit_rate, 4),
        "mrr": round(mrr, 4),
        "avg_precision_at_k": round(avg_precision, 4),
        "top_k": top_k,
        "total_queries": len(results)
    }


# Run evaluation with router
print("Running evaluation WITH Query Router...")
print("Strategy: Hybrid BM25 + Dense + Query Router\n")

results_router, metrics_router = evaluate_retrieval_with_router(
    GOLDEN_QUERIES, top_k=5, alpha=0.5
)

print("="*50)
print("METRICS WITH QUERY ROUTER")
print("="*50)
print(f"Hit Rate@{metrics_router['top_k']}     : {metrics_router['hit_rate']}")
print(f"MRR              : {metrics_router['mrr']}")
print(f"Avg Precision@{metrics_router['top_k']} : {metrics_router['avg_precision_at_k']}")

print("\n")
print("="*50)
print("IMPROVEMENT OVER BASELINE")
print("="*50)
print(f"Hit Rate@5     : 0.4667 → {metrics_router['hit_rate']} (+{round(metrics_router['hit_rate'] - 0.4667, 4)})")
print(f"MRR            : 0.3222 → {metrics_router['mrr']} (+{round(metrics_router['mrr'] - 0.3222, 4)})")
print(f"Avg Precision@5: 0.2267 → {metrics_router['avg_precision_at_k']} (+{round(metrics_router['avg_precision_at_k'] - 0.2267, 4)})")

print("\nPer query breakdown:")
print("-"*70)
for r in results_router:
    status = "✓" if r["hit"] else "✗"
    routing_correct = "→" if r["routed_section"] == r["expected_section"] else "✗route"
    print(f"{status} {r['id']} | {r['ticker']} | expected={r['expected_section']} | routed={r['routed_section']} {routing_correct} | RR={r['reciprocal_rank']:.2f}")

Running evaluation WITH Query Router...
Strategy: Hybrid BM25 + Dense + Query Router

METRICS WITH QUERY ROUTER
Hit Rate@5     : 0.6571
MRR              : 0.6571
Avg Precision@5 : 0.3829


IMPROVEMENT OVER BASELINE
Hit Rate@5     : 0.4667 → 0.6571 (+0.1904)
MRR            : 0.3222 → 0.6571 (+0.3349)
Avg Precision@5: 0.2267 → 0.3829 (+0.1562)

Per query breakdown:
----------------------------------------------------------------------
✓ Q001 | AAPL | expected=Income Statement | routed=Income Statement → | RR=1.00
✓ Q002 | AAPL | expected=Income Statement | routed=Income Statement → | RR=1.00
✓ Q003 | AAPL | expected=Income Statement | routed=Income Statement → | RR=1.00
✓ Q004 | AAPL | expected=Income Statement | routed=Income Statement → | RR=1.00
✗ Q005 | AAPL | expected=MD&A | routed=Any ✗route | RR=0.00
✗ Q006 | AAPL | expected=MD&A | routed=Any ✗route | RR=0.00
✗ Q007 | AAPL | expected=MD&A | routed=MD&A → | RR=0.00
✗ Q008 | AAPL | expected=MD&A | routed=Income Statement ✗route | RR

Running evaluation WITH Query Router...
Strategy: Hybrid BM25 + Dense + Query Router

==================================================
METRICS WITH QUERY ROUTER
==================================================
Hit Rate@5     : 0.8
MRR              : 0.8
Avg Precision@5 : 0.4533


==================================================
IMPROVEMENT OVER BASELINE
==================================================
Hit Rate@5     : 0.4667 → 0.8 (+0.3333)
MRR            : 0.3222 → 0.8 (+0.4778)
Avg Precision@5: 0.2267 → 0.4533 (+0.2266)

Per query breakdown:
----------------------------------------------------------------------
✓ G001 | AAPL | expected=Income Statement | routed=Income Statement → | RR=1.00
✓ G002 | AAPL | expected=Income Statement | routed=Income Statement → | RR=1.00
✓ G003 | AAPL | expected=Income Statement | routed=Income Statement → | RR=1.00
✓ G004 | AAPL | expected=Income Statement | routed=Income Statement → | RR=1.00
✓ G005 | AAPL | expected=Income Statement | routed=Income Statement → | RR=1.00
✓ G006 | AAPL | expected=Balance Sheet | routed=Balance Sheet → | RR=1.00
✓ G007 | AAPL | expected=Balance Sheet | routed=Balance Sheet → | RR=1.00
✓ G008 | AAPL | expected=MD&A | routed=MD&A → | RR=1.00
✓ G009 | AAPL | expected=MD&A | routed=MD&A → | RR=1.00
✗ G010 | AAPL | expected=MD&A | routed=Any ✗route | RR=0.00
✗ G011 | MSFT | expected=Income Statement | routed=Income Statement → | RR=0.00
✗ G012 | MSFT | expected=Income Statement | routed=Income Statement → | RR=0.00
✓ G013 | AAPL | expected=Risk Factors | routed=Risk Factors → | RR=1.00
✓ G014 | AAPL | expected=Cash Flow | routed=Cash Flow → | RR=1.00
✓ G015 | AAPL | expected=Notes | routed=Notes → | RR=1.00

In [105]:
import re

def split_into_sentences(text):
    """
    Split prose text into sentences.
    Returns list of sentences.
    """
    # Split on period, exclamation, question mark followed by space and capital
    sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z])', text.strip())
    # Filter empty
    sentences = [s.strip() for s in sentences if s.strip()]
    return sentences

# Test
test_text = """iPhone net sales increased during the second quarter of 2026. 
This was due to higher net sales of Pro models. 
The strength in foreign currencies had a favorable impact on results."""

sentences = split_into_sentences(test_text)
print(f"Sentences found: {len(sentences)}")
for i, s in enumerate(sentences, 1):
    print(f"  {i}: {s}")

Sentences found: 3
  1: iPhone net sales increased during the second quarter of 2026.
  2: This was due to higher net sales of Pro models.
  3: The strength in foreign currencies had a favorable impact on results.


In [108]:
def is_meaningful_row(row):
    """
    Returns True if row contains actual financial data.
    Filters out: date headers, section labels, empty rows.
    """
    cells = [c.strip() for c in row.split('|') if c.strip()]
    
    if not cells:
        return False
    
    # Must have at least 2 non-empty cells
    non_empty = [c for c in cells if c and c != '-']
    if len(non_empty) < 2:
        return False
    
    # Filter out date rows — contain March/June/September/December/year only
    date_patterns = ['march', 'june', 'september', 'december', 'january', 
                     'february', 'april', 'july', 'august', 'october', 'november']
    row_lower = row.lower()
    if any(month in row_lower for month in date_patterns):
        return False
    
    # Must contain financial data — $ sign or comma-formatted number
    has_financial = any(
        '$' in cell or 
        (any(char.isdigit() for char in cell) and ',' in cell)
        for cell in cells
    )
    
    return has_financial



# Update split_table_into_rows to use filter
def split_table_into_rows(table_markdown):
    lines = table_markdown.strip().split('\n')
    
    header = None
    separator = None
    data_rows = []
    
    for line in lines:
        line = line.strip()
        if not line:
            continue
        if re.match(r'^\|[-|\s:]+\|$', line):
            separator = line
            continue
        if header is None:
            header = line
            continue
        if line.startswith('|'):
            data_rows.append(line)
    
    children = []
    for row in data_rows:
        # Filter out meaningless rows
        if not is_meaningful_row(row):
            continue
        
        child_content = f"{header}\n{separator}\n{row}" if separator else f"{header}\n{row}"
        children.append(child_content)
    
    return children


# Test again
rows = split_table_into_rows(sample_table)
print(f"Meaningful rows found: {len(rows)}")
print("\nSample row children:")
for i, row in enumerate(rows[:3], 1):
    print(f"\nChild {i}:")
    print(row)

Meaningful rows found: 18

Sample row children:

Child 1:
|                                              | Three Months Ended   | Three Months Ended   | Six Months Ended   | Six Months Ended   |
|----------------------------------------------|----------------------|----------------------|--------------------|--------------------|
| Products                                     | $ 80,208             | $ 68,714             | $ 193,951          | $ 166,674          |

Child 2:
|                                              | Three Months Ended   | Three Months Ended   | Six Months Ended   | Six Months Ended   |
|----------------------------------------------|----------------------|----------------------|--------------------|--------------------|
| Services                                     | 30,976               | 26,645               | 60,989             | 52,985             |

Child 3:
|                                              | Three Months Ended   | Three Months Ended   | Six M

In [109]:
import uuid
import json

def create_parent_child_chunks(ticker, result, filing_metadata):
    parents = {}      # docstore — parent_id → parent content
    children = []     # Pinecone — child chunks with parent_id
    
    current_section = "General"
    doc = result.document
    child_index = 0

    for element, level in doc.iterate_items():
        page_no = get_page_no(element)

        # Skip page headers and footers
        if hasattr(element, 'label') and element.label in [
            DocItemLabel.PAGE_HEADER,
            DocItemLabel.PAGE_FOOTER
        ]:
            continue

        # Section detection
        if hasattr(element, 'label') and element.label in [
            DocItemLabel.SECTION_HEADER, DocItemLabel.TITLE
        ]:
            if hasattr(element, 'text'):
                detected = detect_section(element.text)
                if detected != "General":
                    current_section = detected
            continue

        # --- TABLE ---
        if hasattr(element, 'data'):
            table_text = element.export_to_markdown(doc=doc)
            if not table_text.strip():
                continue

            # Create parent
            parent_id = str(uuid.uuid4())[:16]
            parents[parent_id] = {
                "content": table_text,
                "metadata": {
                    **filing_metadata,
                    "chunk_type": "table",
                    "section": current_section,
                    "page": page_no
                }
            }

            # Create children — one row per child
            row_children = split_table_into_rows(table_text)
            for row in row_children:
                children.append({
                    "content": row,
                    "metadata": {
                        **filing_metadata,
                        "parent_id": parent_id,
                        "chunk_type": "table",
                        "section": current_section,
                        "page": page_no,
                        "child_index": child_index
                    }
                })
                child_index += 1
            continue

        # --- PROSE ---
        if hasattr(element, 'text') and element.text.strip():
            prose_text = element.text.strip()

            # Create parent
            parent_id = str(uuid.uuid4())[:16]
            parents[parent_id] = {
                "content": prose_text,
                "metadata": {
                    **filing_metadata,
                    "chunk_type": "prose",
                    "section": current_section,
                    "page": page_no
                }
            }

            # Create children — two sentences per child
            sentences = split_into_sentences(prose_text)
            
            # Group into pairs
            for i in range(0, len(sentences), 2):
                pair = sentences[i:i+2]
                child_text = " ".join(pair)
                if child_text.strip():
                    children.append({
                        "content": child_text,
                        "metadata": {
                            **filing_metadata,
                            "parent_id": parent_id,
                            "chunk_type": "prose",
                            "section": current_section,
                            "page": page_no,
                            "child_index": child_index
                        }
                    })
                    child_index += 1

    return parents, children


# Run for all filings
all_parents = {}
all_children = {}

for filing in FILINGS:
    ticker = filing["ticker"]
    result = all_results[ticker]["result"]
    filing_metadata = all_results[ticker]["metadata"]
    
    parents, children = create_parent_child_chunks(
        ticker, result, filing_metadata
    )
    
    all_parents[ticker] = parents
    all_children[ticker] = children
    
    print(f"{ticker}:")
    print(f"  Parents : {len(parents)}")
    print(f"  Children: {len(children)}")

AAPL:
  Parents : 199
  Children: 447
MSFT:
  Parents : 468
  Children: 1011


In [110]:
import json
import os

# Combine all parents into one docstore
docstore = {}
for ticker, parents in all_parents.items():
    docstore.update(parents)

# Save to disk
docstore_path = "../data/processed/docstore.json"
os.makedirs("../data/processed", exist_ok=True)

with open(docstore_path, "w") as f:
    json.dump(docstore, f, indent=2)

print(f"Docstore saved: {docstore_path}")
print(f"Total parents: {len(docstore)}")

# Verify load works
with open(docstore_path, "r") as f:
    loaded = json.load(f)

print(f"Docstore loaded: {len(loaded)} parents")
print(f"Sample parent_id: {list(loaded.keys())[0]}")

Docstore saved: ../data/processed/docstore.json
Total parents: 667
Docstore loaded: 667 parents
Sample parent_id: 76fee7d7-d1db-41


In [111]:
# Refit BM25 on children corpus
all_child_texts = []
for ticker, children in all_children.items():
    for child in children:
        all_child_texts.append(child["content"])

print(f"Total children to fit BM25: {len(all_child_texts)}")

bm25_encoder_pc = BM25Encoder()
bm25_encoder_pc.fit(all_child_texts)
print("BM25Encoder fitted on children corpus")

Total children to fit BM25: 1458


  0%|          | 0/1458 [00:00<?, ?it/s]

BM25Encoder fitted on children corpus


In [158]:
def upsert_children_hybrid(ticker, children, namespace_suffix="_PC"):
    namespace = f"{ticker}{namespace_suffix}"
    print(f"\nUpserting {ticker} children to namespace: {namespace}")
    print(f"Total children: {len(children)}")
    
    skipped = 0
    for i in tqdm(range(0, len(children), BATCH_SIZE)):
        batch = children[i:i + BATCH_SIZE]
        texts = [c["content"] for c in batch]
        
        # Dense vectors
        dense_embeddings = get_embeddings(texts)
        
        # Sparse vectors
        sparse_embeddings = bm25_encoder_pc.encode_documents(texts)
        
        vectors = []
        for child, dense, sparse in zip(batch, dense_embeddings, sparse_embeddings):
            if not sparse["values"]:
                skipped += 1
                continue
            
            chunk_id = make_chunk_id(ticker, child["metadata"]["child_index"])
            vectors.append({
                "id": chunk_id,
                "values": dense,
                "sparse_values": sparse,
                "metadata": {
                    **child["metadata"],
                    "content": child["content"][:1000]
                }
            })
        
        if vectors:
            index.upsert(vectors=vectors, namespace=namespace)
    
    print(f"  ✓ {ticker} children upserted | skipped: {skipped}")


# Upsert all children
for ticker, children in all_children.items():
    upsert_children_hybrid(ticker, children)

print("\nFinal index stats:")
print(index.describe_index_stats())


Upserting AAPL children to namespace: AAPL_PC
Total children: 447


100%|██████████| 5/5 [00:17<00:00,  3.50s/it]


  ✓ AAPL children upserted | skipped: 2

Upserting MSFT children to namespace: MSFT_PC
Total children: 1011


100%|██████████| 11/11 [00:30<00:00,  2.73s/it]


  ✓ MSFT children upserted | skipped: 1

Final index stats:
{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'dotproduct',
 'namespaces': {'AAPL': {'vector_count': 265},
                'AAPL_PC': {'vector_count': 445},
                'MSFT': {'vector_count': 662},
                'MSFT_PC': {'vector_count': 1010}},
 'total_vector_count': 2382,
 'vector_type': 'dense'}


In [159]:
def retrieve_with_parent_child(query, ticker, section=None, chunk_type=None, top_k=5, alpha=0.5):
    namespace = f"{ticker}_PC"
    
    # Encode query
    dense_vector = get_embeddings([query])[0]
    sparse_vector = bm25_encoder_pc.encode_queries(query)
    
    # Build filter
    filter_dict = {"section": {"$nin": NOISE_SECTIONS}}
    if section:
        filter_dict["section"] = {"$eq": section}
    if chunk_type:
        filter_dict["chunk_type"] = {"$eq": chunk_type}
    
    # Search children
    results = index.query(
        vector=scale_dense(dense_vector, alpha),
        sparse_vector=scale_sparse(sparse_vector, alpha),
        top_k=top_k,
        namespace=namespace,
        include_metadata=True,
        filter=filter_dict
    )
    
    # Fetch parents from docstore
    seen_parents = set()
    retrieved = []
    
    for match in results.matches:
        parent_id = match.metadata.get("parent_id")
        
        if not parent_id or parent_id in seen_parents:
            continue
        
        seen_parents.add(parent_id)
        
        parent = docstore.get(parent_id)
        if parent:
            retrieved.append({
                "child_score": match.score,
                "child_content": match.metadata.get("content"),
                "parent_content": parent["content"],
                "metadata": {
                    **match.metadata,
                    "parent_id": parent_id
                }
            })
    
    return retrieved


# Test
query = "What was Apple total net sales in Q2 2026?"
results = retrieve_with_parent_child(
    query, "AAPL",
    section="Income Statement",
    chunk_type="table",
    top_k=5
)



In [160]:
print(f"Query: {query}\n")
for i, r in enumerate(results, 1):
    print(f"Result {i} — child score: {r['child_score']:.4f}")
    print(f"\nChild content (full):")
    print(r['child_content'])
    print(f"\nParent content (first 400 chars):")
    print(r['parent_content'][:400])
    print("\n" + "="*60)

Query: What was Apple total net sales in Q2 2026?

Result 1 — child score: 0.3207

Child content (full):
|                                              | Three Months Ended   | Three Months Ended   | Six Months Ended   | Six Months Ended   |
|----------------------------------------------|----------------------|----------------------|--------------------|--------------------|
| Total net sales                              | 111,184              | 95,359               | 254,940            | 219,659            |

Parent content (first 400 chars):
|                                              | Three Months Ended   | Three Months Ended   | Six Months Ended   | Six Months Ended   |
|----------------------------------------------|----------------------|----------------------|--------------------|--------------------|
|                                              | March 28, 2026       | March 29, 2025       | March 28, 2026     | March 29, 

Result 2 — child score: 0.2334

Child content (

In [116]:
def retrieve_pc_with_router(query, ticker, top_k=5, alpha=0.5):
    # Route query
    route = route_query(query)
    section = route["section"]
    chunk_type = route["chunk_type"]
    
    # Retrieve with parent-child
    if section == "Any":
        results = retrieve_with_parent_child(
            query=query,
            ticker=ticker,
            chunk_type=chunk_type,
            top_k=top_k,
            alpha=alpha
        )
    else:
        results = retrieve_with_parent_child(
            query=query,
            ticker=ticker,
            section=section,
            chunk_type=chunk_type,
            top_k=top_k,
            alpha=alpha
        )
    
    return results, route


# Quick test
query = "What was Apple total net sales in Q2 2026?"
results, route = retrieve_pc_with_router(query, "AAPL", top_k=3)

print(f"Query : {query}")
print(f"Route : {route}")
print(f"Results: {len(results)}\n")

for i, r in enumerate(results, 1):
    print(f"Result {i} | score: {r['child_score']:.4f} | section: {r['metadata'].get('section')} | page: {r['metadata'].get('page')}")
    print(f"  Child: {r['child_content'][:100]}")

Query : What was Apple total net sales in Q2 2026?
Route : {'chunk_type': 'table', 'section': 'Income Statement'}
Results: 1

Result 1 | score: 0.3207 | section: Income Statement | page: 4.0
  Child: |                                              | Three Months Ended   | Three Months Ended   | Six M


In [161]:
def evaluate_parent_child(golden_queries, top_k=5, alpha=0.5):
    
    results = []
    
    for gq in golden_queries:
        matches, route = retrieve_pc_with_router(
            query=gq["query"],
            ticker=gq["ticker"],
            top_k=top_k,
            alpha=alpha
        )
        
        hit = False
        reciprocal_rank = 0
        relevant_count = 0
        
        for rank, match in enumerate(matches, 1):
            section_match = match["metadata"].get("section") == gq["expected_section"]
            type_match = match["metadata"].get("chunk_type") == gq["expected_chunk_type"]
            
            is_relevant = section_match and type_match
            
            if is_relevant:
                relevant_count += 1
                if not hit:
                    hit = True
                    reciprocal_rank = 1 / rank
        
        precision_at_k = relevant_count / top_k
        
        results.append({
            "id": gq["id"],
            "query": gq["query"],
            "ticker": gq["ticker"],
            "expected_section": gq["expected_section"],
            "expected_type": gq["expected_chunk_type"],
            "routed_section": route["section"],
            "hit": hit,
            "reciprocal_rank": reciprocal_rank,
            "precision_at_k": precision_at_k
        })
    
    hit_rate = sum(r["hit"] for r in results) / len(results)
    mrr = sum(r["reciprocal_rank"] for r in results) / len(results)
    avg_precision = sum(r["precision_at_k"] for r in results) / len(results)
    
    return results, {
        "hit_rate": round(hit_rate, 4),
        "mrr": round(mrr, 4),
        "avg_precision_at_k": round(avg_precision, 4),
        "top_k": top_k,
        "total_queries": len(results)
    }



In [162]:
print("Running Layer 1.5 evaluation after router fixes...")
results_pc, metrics_pc = evaluate_parent_child(GOLDEN_QUERIES, top_k=3, alpha=0.5)

print("="*60)
print("COMPLETE COMPARISON")
print("="*60)
print(f"{'Metric':<20} {'Baseline':>10} {'Layer 1':>10} {'Layer 1.5':>10}")
print("-"*60)

baseline = {"hit_rate": 0.4667, "mrr": 0.3222, "avg_precision_at_k": 0.2267}
layer1   = {"hit_rate": 0.6571, "mrr": 0.6571, "avg_precision_at_k": 0.3829}

for metric in ["hit_rate", "mrr", "avg_precision_at_k"]:
    b  = baseline[metric]
    l1 = layer1[metric]
    l15 = metrics_pc[metric]
    print(f"{metric:<20} {b:>10.4f} {l1:>10.4f} {l15:>10.4f}")

print("\nPer query breakdown:")
print("-"*70)
for r in results_pc:
    status = "✓" if r["hit"] else "✗"
    routing = "→" if r["routed_section"] == r["expected_section"] else "✗route"
    print(f"{status} {r['id']} | {r['expected_section']} | routed={r['routed_section']} {routing} | RR={r['reciprocal_rank']:.2f}")

Running Layer 1.5 evaluation after router fixes...
COMPLETE COMPARISON
Metric                 Baseline    Layer 1  Layer 1.5
------------------------------------------------------------
hit_rate                 0.4667     0.6571     0.7143
mrr                      0.3222     0.6571     0.7143
avg_precision_at_k       0.2267     0.3829     0.3905

Per query breakdown:
----------------------------------------------------------------------
✓ Q001 | Income Statement | routed=Income Statement → | RR=1.00
✓ Q002 | Income Statement | routed=Income Statement → | RR=1.00
✓ Q003 | Income Statement | routed=Income Statement → | RR=1.00
✓ Q004 | Income Statement | routed=Income Statement → | RR=1.00
✗ Q005 | MD&A | routed=MD&A → | RR=0.00
✗ Q006 | MD&A | routed=Income Statement ✗route | RR=0.00
✗ Q007 | MD&A | routed=MD&A → | RR=0.00
✗ Q008 | MD&A | routed=MD&A → | RR=0.00
✓ Q009 | Notes | routed=Notes → | RR=1.00
✗ Q010 | MD&A | routed=MD&A → | RR=0.00
✓ Q011 | Balance Sheet | routed=Balance Shee

In [139]:
# Save BM25 encoder to disk
bm25_encoder.dump("../data/processed/bm25_encoder.json")
print("BM25 encoder saved")

BM25 encoder saved


In [163]:
# Run in Layer 1 notebook
print("Running final evaluation after all fixes...")
results_pc, metrics_pc = evaluate_parent_child(GOLDEN_QUERIES, top_k=5, alpha=0.5)

print("="*60)
print("FINAL COMPARISON — ALL LAYERS")
print("="*60)
print(f"{'Metric':<20} {'Baseline':>10} {'Layer 1':>10} {'Layer 1.5':>10}")
print("-"*60)

baseline = {"hit_rate": 0.4667, "mrr": 0.3222, "avg_precision_at_k": 0.2267}
layer1   = {"hit_rate": 0.6571, "mrr": 0.6571, "avg_precision_at_k": 0.3829}

for metric in ["hit_rate", "mrr", "avg_precision_at_k"]:
    b   = baseline[metric]
    l1  = layer1[metric]
    l15 = metrics_pc[metric]
    pct = round(((l15 - l1) / l1) * 100, 1)
    print(f"{metric:<20} {b:>10.4f} {l1:>10.4f} {l15:>10.4f}  ({pct:+.1f}%)")

print("\nPer query breakdown:")
print("-"*70)
for r in results_pc:
    status = "✓" if r["hit"] else "✗"
    routing = "→" if r["routed_section"] == r["expected_section"] else "✗route"
    print(f"{status} {r['id']} | {r['expected_section']:<25} | routed={r['routed_section']:<20} {routing} | RR={r['reciprocal_rank']:.2f}")

Running final evaluation after all fixes...
FINAL COMPARISON — ALL LAYERS
Metric                 Baseline    Layer 1  Layer 1.5
------------------------------------------------------------
hit_rate                 0.4667     0.6571     0.7143  (+8.7%)
mrr                      0.3222     0.6571     0.7143  (+8.7%)
avg_precision_at_k       0.2267     0.3829     0.3371  (-12.0%)

Per query breakdown:
----------------------------------------------------------------------
✓ Q001 | Income Statement          | routed=Income Statement     → | RR=1.00
✓ Q002 | Income Statement          | routed=Income Statement     → | RR=1.00
✓ Q003 | Income Statement          | routed=Income Statement     → | RR=1.00
✓ Q004 | Income Statement          | routed=Income Statement     → | RR=1.00
✗ Q005 | MD&A                      | routed=MD&A                 → | RR=0.00
✗ Q006 | MD&A                      | routed=Income Statement     ✗route | RR=0.00
✗ Q007 | MD&A                      | routed=MD&A             